# Hagemi: refine the hair behind her tail

Repaints the hidden hair with [Animagine XL 4.0](https://huggingface.co/cagliostrolab/animagine-xl-4.0) (CreativeML Open RAIL++-M), starting from the existing LaMa result so the hair keeps its shape.

1. **Runtime > Change runtime type > GPU** (T4 is enough).
2. **Runtime > Run all**.
3. When asked, upload the crops zip made by `python tools/inpaint_under.py --export crops.zip`.
4. A results zip downloads when it finishes. Bring it back with `python tools/inpaint_under.py --import results.zip --variant s050_a`.

Only pixels inside each mask are ever used, and only pixels her tail fully covers are in the mask, so nothing visible at rest can change.

In [ ]:
%pip install -q diffusers==0.42.1 transformers==5.22.1 accelerate==1.15.0 safetensors==0.8.0

In [ ]:
import os

MODEL = os.environ.get('HAGEMI_MODEL', 'cagliostrolab/animagine-xl-4.0')
LONGEST = int(os.environ.get('HAGEMI_LONGEST', 1024))  # Crops are enlarged to this for the model.
STRENGTHS = [0.5, 0.65]  # Lower keeps more of the existing paint; higher redraws more freely.
SEEDS = {'a': 20261105, 'b': 20261106, 'c': 20261107, 'd': 20261108}  # Every strength runs with every seed.
STEPS = int(os.environ.get('HAGEMI_STEPS', 30))
GUIDANCE = 5
QUALITY = 'safe, flat color, soft shading, clean lineart, thin dark outline, simple background, white background, masterpiece, high score, great score, absurdres'
HAIR = '1girl, solo, chibi, very long hair, wavy hair, blue hair, purple hair, gradient hair, pink hair, hair focus, '
SKIRT = '1girl, solo, chibi, close-up, pleated skirt, layered skirt, indigo skirt, blue-violet fabric, plaid pattern, thin pale lines on fabric, skirt focus, '
# What each hole hides, read from the poses. Anything unlisted is skirt, which most holes beside the gems are.
PROMPTS = {'greet_noon': HAIR, 'greet_evening': HAIR + 'pleated skirt, '}
# The tail must not come back, and a gem or object painted in the hole is the mistake to avoid.
NEGATIVE = ('tail, cat tail, animal tail, fur, gem, crystal, jewelry, star, object, brown, grey, white patch, hole, '
            'lowres, bad anatomy, text, error, cropped, worst quality, low quality, '
            'low score, bad score, average score, signature, watermark, username, blurry, 3d, realistic')

In [ ]:
import shutil
import zipfile

try:
    from google.colab import files
except ImportError:
    files = None

if files:
    crops_zip = next(iter(files.upload()))
else:
    crops_zip = os.environ['HAGEMI_CROPS']

shutil.rmtree('crops', ignore_errors=True)
with zipfile.ZipFile(crops_zip) as archive:
    archive.extractall('crops')
print(len(os.listdir('crops')) - 1, 'poses to refine')

In [ ]:
import torch
from diffusers import EulerAncestralDiscreteScheduler, StableDiffusionXLInpaintPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu' and files:
    raise RuntimeError('No GPU. Choose Runtime > Change runtime type > GPU, then run again.')
dtype = torch.float16 if device == 'cuda' else torch.float32
pipe = StableDiffusionXLInpaintPipeline.from_pretrained(MODEL, torch_dtype=dtype, use_safetensors=True, add_watermarker=False).to(device)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)
print('loaded', MODEL, 'on', device)

In [ ]:
import json
import time

import numpy as np
from IPython.display import display
from PIL import Image, ImageDraw

manifest = json.load(open('crops/manifest.json'))
shutil.rmtree('results', ignore_errors=True)
os.makedirs('results')
shutil.copy('crops/manifest.json', 'results/manifest.json')


def fit(width, height):
    """Longest side becomes LONGEST, both rounded to the multiple of 8 the model needs."""
    scale = LONGEST / max(width, height)
    return max(8, round(width * scale / 8) * 8), max(8, round(height * scale / 8) * 8)


tiles = []
started = time.time()
for index, pose in enumerate(manifest):
    init = Image.open(f'crops/{pose}/init.png').convert('RGB')
    mask = Image.open(f'crops/{pose}/mask.png').convert('L')
    size = fit(*init.size)
    big_init = init.resize(size, Image.LANCZOS)
    big_mask = mask.resize(size, Image.BILINEAR)
    row = [init]
    os.makedirs(f'results/{pose}')
    prompt = PROMPTS.get(pose, SKIRT) + QUALITY
    for strength in STRENGTHS:
        for name, seed in SEEDS.items():
            generator = torch.Generator(device).manual_seed(seed)
            out = pipe(prompt=prompt, negative_prompt=NEGATIVE, image=big_init, mask_image=big_mask, width=size[0], height=size[1],
                       strength=strength, num_inference_steps=STEPS, guidance_scale=GUIDANCE, generator=generator).images[0]
            if np.asarray(out).mean() < 2:
                raise RuntimeError(f'{pose} came back black, so the VAE overflowed. Tell Claude.')
            out = out.resize(init.size, Image.LANCZOS)
            out.save(f'results/{pose}/s{round(strength * 100):03d}_{name}.png')
            row.append(out)
    tiles.append((pose, row))
    print(f'{index + 1}/{len(manifest)} {pose} ({time.time() - started:.0f}s)')

In [ ]:
# A sheet to look at here, also included in the results: existing paint, then each strength.
cell = 220
sheet = Image.new('RGB', (cell * (1 + len(STRENGTHS) * len(SEEDS)), cell * len(tiles)), 'white')
draw = ImageDraw.Draw(sheet)
for r, (pose, row) in enumerate(tiles):
    for c, image in enumerate(row):
        k = cell / max(image.size)
        sheet.paste(image.resize((int(image.width * k), int(image.height * k)), Image.LANCZOS), (c * cell, r * cell))
    draw.text((4, r * cell + 2), pose, fill='black')
sheet.save('results/preview.png')
display(sheet)

results_zip = shutil.make_archive('hagemi-inpaint-results', 'zip', 'results')
print('wrote', results_zip)
if files:
    files.download(results_zip)